# Rodar RNAMining local

Notebook simples para:

- rodar o RNAMining nas espécies do conjunto de dados(Conjunto base: S4_File)
- salvar `predictions.txt` por espécie
- ler o ground truth diretamente do FASTA de teste
- calcular métricas finais por espécie

In [ ]:
from pathlib import Path
import subprocess
import csv
import math
import pandas as pd

## 1. Paths principais

In [ ]:
# Pasta base do projeto RNAmining
BASE_RNAMINING = Path("/home/samuel/projects/RNAmining")

# Script oficial do RNAmining
SCRIPT_RNAMINING = BASE_RNAMINING / "volumes" / "rnamining-front" / "assets" / "scripts" / "rnamining.py"

# Pasta com os FASTAs de teste do S4
DADOS_RNAMINING = BASE_RNAMINING / "volumes" / "rnamining-front" / "data" / "S4_File" / "S4_File" / "Model_Organisms"

# Pasta onde os resultados serão salvos
OUT_DIR = BASE_RNAMINING / "rnamining_out"

# Tipo de predição usado pelo RNAmining
PREDICTION_TYPE = "coding_prediction"

## 2. Funções auxiliares

In [ ]:
# Converte o nome do arquivo para o nome do organismo esperado pelo RNAMining
def extrair_nome_organismo(caminho_fasta):
    nome = caminho_fasta.stem.replace("_test", "")
    partes = nome.split("_")
    return "_".join([partes[0].capitalize(), partes[1].lower()])


# Lê o FASTA de teste e monta o ground truth
# Regra usada no S4:
# - cds -> 1
# - ncrna -> 0
def ler_gt_fasta(caminho_fasta):
    gt = {}

    with open(caminho_fasta, "r") as f:
        for linha in f:
            if linha.startswith(">"):
                header = linha[1:].strip()
                seq_id = header.split()[0]
                texto = header.lower()

                if " cds " in f" {texto} ":
                    gt[seq_id] = 1
                elif " ncrna " in f" {texto} ":
                    gt[seq_id] = 0

    return gt


# Normaliza a label prevista
def normalizar_label_predicao(label):
    texto = str(label).strip().lower()

    if texto in {"coding", "coding_rna", "cod", "mrna"}:
        return 1
    elif texto in {"non-coding", "noncoding", "noncodingrna", "lncrna", "ncrna"}:
        return 0
    else:
        return None


# Lê o predictions.txt do RNAMining
# Para casar com o GT, usamos apenas o primeiro token do header
def ler_predicoes_rnamining(caminho_pred):
    pred = {}

    with open(caminho_pred, "r") as f:
        linhas = [linha.rstrip("\n") for linha in f]

    for linha in linhas:
        linha = linha.strip()

        if not linha:
            continue
        if linha.startswith("RNAMining Predictions"):
            continue
        if linha.startswith("Prediction Type:"):
            continue
        if linha.startswith("Name of the Organism:"):
            continue
        if linha.startswith("Sequence ID"):
            continue

        partes = linha.split("\t")

        if len(partes) < 2:
            continue

        header_original = partes[0].strip()
        seq_id = header_original.split()[0]

        label_prevista = partes[1].strip()
        label_prevista = normalizar_label_predicao(label_prevista)

        if label_prevista is not None:
            pred[seq_id] = label_prevista

    return pred


# Calcula TP, TN, FP, FN e métricas
def calcular_metricas(gt, pred):
    ids_comuns = sorted(set(gt) & set(pred))

    tp = tn = fp = fn = 0

    for seq_id in ids_comuns:
        verdadeiro = gt[seq_id]
        previsto = pred[seq_id]

        if previsto == 1 and verdadeiro == 1:
            tp += 1
        elif previsto == 0 and verdadeiro == 0:
            tn += 1
        elif previsto == 1 and verdadeiro == 0:
            fp += 1
        elif previsto == 0 and verdadeiro == 1:
            fn += 1

    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total != 0 else 0
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    denominador = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denominador if denominador != 0 else 0

    return {
        "n_total_eval": len(gt),
        "n_pred_rows": len(pred),
        "n_merged": len(ids_comuns),
        "n_missing_pred": len(set(gt) - set(pred)),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
    }

## 3. Conferência rápida dos arquivos de teste

In [ ]:
arquivos_teste = sorted(DADOS_RNAMINING.glob("*_test.fa"))
len(arquivos_teste), [arquivo.name for arquivo in arquivos_teste[:5]]

## 4. Rodar o RNAMining

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_EXEC = OUT_DIR / "execucao_rnamining.csv"

with open(CSV_EXEC, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "species",
        "organism_name",
        "status",
        "test_file",
        "output_dir",
        "prediction_file",
        "returncode"
    ])

for caminho_teste in arquivos_teste:
    nome_curto = caminho_teste.stem.replace("_test", "").split("_")[0].lower()
    organism_name = extrair_nome_organismo(caminho_teste)

    pasta_saida = OUT_DIR / nome_curto
    pasta_saida.mkdir(parents=True, exist_ok=True)

    arquivo_pred = pasta_saida / "predictions.txt"

    comando = [
        "python",
        str(SCRIPT_RNAMINING),
        "-f", str(caminho_teste),
        "-organism_name", organism_name,
        "-prediction_type", PREDICTION_TYPE,
        "-output_folder", str(pasta_saida)
    ]

    print(f"Rodando {organism_name}")
    resultado = subprocess.run(comando, capture_output=True, text=True)

    status_exec = "ok" if resultado.returncode == 0 else "erro"

    with open(CSV_EXEC, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            nome_curto,
            organism_name,
            status_exec,
            str(caminho_teste),
            str(pasta_saida),
            str(arquivo_pred),
            resultado.returncode
        ])

## 5. Ver resumo das execuções

In [ ]:
pd.read_csv(CSV_EXEC)

## 6. Calcular métricas por espécie

In [ ]:
CSV_METRICS = OUT_DIR / "metrics_rnamining.csv"

with open(CSV_METRICS, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "species",
        "organism_name",
        "status",
        "n_total_eval",
        "n_pred_rows",
        "n_merged",
        "n_missing_pred",
        "tp",
        "tn",
        "fp",
        "fn",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mcc"
    ])

for caminho_teste in arquivos_teste:
    nome_curto = caminho_teste.stem.replace("_test", "").split("_")[0].lower()
    organism_name = extrair_nome_organismo(caminho_teste)

    caminho_pred = OUT_DIR / nome_curto / "predictions.txt"

    gt = ler_gt_fasta(caminho_teste)
    pred = ler_predicoes_rnamining(caminho_pred)

    metricas = calcular_metricas(gt, pred)

    with open(CSV_METRICS, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            nome_curto,
            organism_name,
            "ok",
            metricas["n_total_eval"],
            metricas["n_pred_rows"],
            metricas["n_merged"],
            metricas["n_missing_pred"],
            metricas["tp"],
            metricas["tn"],
            metricas["fp"],
            metricas["fn"],
            metricas["accuracy"],
            metricas["precision"],
            metricas["recall"],
            metricas["f1"],
            metricas["mcc"]
        ])

## 7. Ver tabela final de métricas

In [ ]:
pd.read_csv(CSV_METRICS)

## 8. Inspeção rápida de uma espécie

In [ ]:
especie_exemplo = "anolis"

caminho_pred = OUT_DIR / especie_exemplo / "predictions.txt"
caminho_teste = DADOS_RNAMINING / "Anolis_carolinensis_test.fa"

print("Arquivo de predição:")
print(caminho_pred)

print("\nPrimeiras linhas do predictions.txt:")
with open(caminho_pred, "r") as f:
    for i, linha in enumerate(f):
        if i == 8:
            break
        print(linha.rstrip())

print("\nResumo do GT e da predição:")
gt_exemplo = ler_gt_fasta(caminho_teste)
pred_exemplo = ler_predicoes_rnamining(caminho_pred)

print("GT:", len(gt_exemplo))
print("Pred:", len(pred_exemplo))
print("IDs em comum:", len(set(gt_exemplo) & set(pred_exemplo)))